# 04 - Gold Layer Analytics

Explore Gold layer business views: Customer 360, AML Summary, and Regulatory Reports.

In [ ]:
import sys
sys.path.insert(0, '/opt/spark-jobs')
from utils.spark_session import create_spark_session
from utils.delta_helpers import read_delta
from pyspark.sql import functions as F

spark = create_spark_session(app_name='GoldAnalytics')

In [ ]:
# Customer 360 overview
c360 = read_delta(spark, 's3a://gold/customer_360')
print(f'Customer 360: {c360.count()} customers')

# Risk distribution
c360.groupBy('risk_category').agg(
    F.count('*').alias('count'),
    F.round(F.avg('total_balance'), 2).alias('avg_balance'),
    F.round(F.avg('total_transactions'), 0).alias('avg_txns'),
    F.sum('aml_alert_count').alias('total_aml_alerts')
).show()

In [ ]:
# Top 10 highest-risk customers
c360.orderBy(F.col('composite_risk_score').desc()).select(
    'customer_id', 'customer_name', 'risk_category',
    'composite_risk_score', 'aml_alert_count', 'total_balance'
).show(10, truncate=False)

In [ ]:
# AML Summary
aml = read_delta(spark, 's3a://gold/aml_summary')
print(f'AML Summary: {aml.count()} records')

aml.groupBy('alert_type').agg(
    F.sum('alert_count').alias('total_alerts'),
    F.round(F.sum('total_flagged_amount'), 2).alias('total_amount'),
    F.round(F.avg('resolution_rate_pct'), 1).alias('avg_resolution_pct')
).show()

In [ ]:
# Regulatory Report sample
reg = read_delta(spark, 's3a://gold/regulatory_reports')
print(f'Regulatory Report: {reg.count()} records')
reg.select(
    'customer_hash', 'report_month', 'risk_category',
    'transaction_count', 'total_amount', 'aml_alert_count', 'regulatory_body'
).show(5, truncate=False)

In [ ]:
spark.stop()